MD# Advisor agent — telecom intent classifier (fine-tuned MiniLM)

**What it learns:** to recognise what a telecom customer wants (26 intents: dispute an invoice, poor signal, roaming, cancel the plan...) from what they say, even when they phrase it in their own words.
**Why the advisor needs it:** today the advisor finds passages with TF-IDF, which only matches words. A fine-tuned model understands paraphrases ("my bill is wrong" = dispute_invoice).
**Data:** [Bitext telco customer-support dataset](https://huggingface.co/datasets/bitext/Bitext-telco-llm-chatbot-training-dataset), 26,000 customer messages, 26 intents, CDLA-Sharing-1.0 (attribution required).
**Model:** `sentence-transformers/all-MiniLM-L6-v2` (22 M parameters, runs on CPU) + a classification head, fine-tuned.

**Run:** GPU (T4 or better) → *Run all*. ~5–10 min. At the end, download **`advisor_intent_model.zip`**.
On Kaggle: *Settings → Internet: On*, *Accelerator: GPU*.

In [1]:
!pip install -q -U transformers datasets scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
import os, json, re, random, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
# Kaggle writes to /kaggle/working, Lightning AI to the current folder
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
print("Outputs go to", OUT)

Outputs go to /content


In [3]:
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
torch.manual_seed(SEED)

df = load_dataset("bitext/Bitext-telco-llm-chatbot-training-dataset", split="train").to_pandas()
df = df[["instruction", "category", "intent", "tags"]].dropna()
df["instruction"] = df["instruction"].str.strip()
before = len(df)
df = df.drop_duplicates("instruction")          # the same sentence must not be in train and test
print(f"{before} rows, {len(df)} after removing exact duplicates")
print(df.intent.value_counts().to_string())

Device: cuda Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/8.98k [00:00<?, ?B/s]

bitext-telco-llm-chatbot-training-datase(…): reconstructing file:   0%|          |  0.00B / 15.1MB            

bitext-telco-llm-chatbot-training-datase(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26000 [00:00<?, ? examples/s]

26000 rows, 25986 after removing exact duplicates
intent
dispute_invoice                        1000
invoices                               1000
get_compensation                       1000
report_problem                         1000
check_excess_data_charges              1000
human_agent                            1000
activate_phone                         1000
activate_roaming                       1000
check_signal_coverage                  1000
payment_methods                        1000
sign_up_for_plan                       1000
cancel_plan                            1000
install_internet                       1000
deactivate_call_management_services    1000
change_plan                            1000
activate_call_management_services       999
check_usage                             999
report_poor_signal_coverage             999
change_provider                         999
deactivate_phone                        999
schedule_payments                       999
check_mobile_paymen

MD## Split 80 / 10 / 10 (stratified by intent)

In [4]:
labels = sorted(df.intent.unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df["label"] = df.intent.map(label2id)

train_df, rest = train_test_split(df, test_size=0.2, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(rest, test_size=0.5, stratify=rest.label, random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

20788 2599 2599


MD## Fine-tuning (plain PyTorch loop, no Trainer: works with any transformers version)

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

BASE = "sentence-transformers/all-MiniLM-L6-v2"
MAX_LEN, BATCH, EPOCHS, LR = 64, 64, 4, 5e-5

tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=len(labels), id2label=id2label, label2id=label2id).to(DEVICE)

def batches(frame, shuffle):
    idx = np.arange(len(frame))
    if shuffle: np.random.shuffle(idx)
    texts, ys = frame.instruction.values, frame.label.values
    for i in range(0, len(idx), BATCH):
        j = idx[i:i + BATCH]
        enc = tok(list(texts[j]), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        yield enc, torch.tensor(ys[j]).to(DEVICE)

@torch.no_grad()
def predict(texts, bs=256):
    model.eval(); out = []
    for i in range(0, len(texts), bs):
        enc = tok(list(texts[i:i + bs]), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        out.append(torch.softmax(model(**enc).logits, -1).cpu().numpy())
    return np.vstack(out)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * ((len(train_df) + BATCH - 1) // BATCH)
sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")

best, best_state = -1, None
for epoch in range(EPOCHS):
    model.train(); t0 = time.time(); total = 0
    for enc, y in batches(train_df, True):
        with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
            loss = model(**enc, labels=y).loss
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
    val_pred = predict(val_df.instruction.values).argmax(1)
    f1 = f1_score(val_df.label, val_pred, average="macro")
    print(f"epoch {epoch+1}: loss {total:.1f}  val macro-F1 {f1:.4f}  ({time.time()-t0:.0f}s)")
    if f1 > best:
        best, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_1907/995000088.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")


epoch 1: loss 496.1  val macro-F1 0.9969  (14s)
epoch 2: loss 46.4  val macro-F1 0.9988  (12s)
epoch 3: loss 17.5  val macro-F1 0.9996  (13s)
epoch 4: loss 12.1  val macro-F1 0.9996  (13s)


<All keys matched successfully>

MD## Evaluation 1 — held-out test split (same distribution as training)

In [6]:
test_prob = predict(test_df.instruction.values)
test_pred = test_prob.argmax(1)
acc = accuracy_score(test_df.label, test_pred); f1 = f1_score(test_df.label, test_pred, average="macro")
print(f"TEST accuracy {acc:.4f}   macro-F1 {f1:.4f}")
print(classification_report(test_df.label, test_pred, target_names=labels, digits=3))

TEST accuracy 0.9996   macro-F1 0.9996
                                     precision    recall  f1-score   support

  activate_call_management_services      1.000     1.000     1.000       100
                     activate_phone      1.000     1.000     1.000       100
                   activate_roaming      1.000     1.000     1.000       100
                        cancel_plan      1.000     0.990     0.995       100
                        change_plan      0.990     1.000     0.995       100
                    change_provider      1.000     1.000     1.000       100
             check_cancellation_fee      1.000     1.000     1.000       100
          check_excess_data_charges      1.000     1.000     1.000       100
              check_mobile_payments      1.000     1.000     1.000       100
              check_signal_coverage      1.000     1.000     1.000       100
                        check_usage      1.000     1.000     1.000       100
                   customer_service 

MD## Evaluation 2 — messages written by hand, in the style of our call centre (harder, out of distribution)
Bitext sentences are variations of templates, so the first score is optimistic. These 30 sentences were written separately; this is the honest score to quote.

In [7]:
HARD = [
 ("I was charged twice this month and I want my money back", "dispute_invoice"),
 ("there is a charge on my bill I never agreed to", "dispute_invoice"),
 ("can you send me a copy of last month's bill", "invoices"),
 ("where do I find my invoices", "invoices"),
 ("this is the third time I call and nobody fixes my problem, I want compensation", "get_compensation"),
 ("I deserve a refund for the days without service", "get_compensation"),
 ("no network at all in my house since Monday", "report_poor_signal_coverage"),
 ("calls keep dropping when I'm at work", "report_poor_signal_coverage"),
 ("my internet stopped working this morning", "report_problem"),
 ("why did I pay extra for data this month", "check_excess_data_charges"),
 ("how much data do I have left", "check_usage"),
 ("I want to put a cap so I never go over my limit", "set_usage_limits"),
 ("let me talk to a real person please", "human_agent"),
 ("what number do I call for customer service", "customer_service"),
 ("which cards can I use to pay", "payment_methods"),
 ("I want to pay my bill now", "pay"),
 ("can I set up automatic monthly payments", "schedule_payments"),
 ("I'm travelling to Italy next week, will my phone work there", "activate_roaming"),
 ("I want to turn on call forwarding", "activate_call_management_services"),
 ("I just got a new SIM, how do I activate it", "activate_phone"),
 ("my phone was stolen, block my line", "deactivate_phone"),
 ("I need internet installed in my new flat", "install_internet"),
 ("is there coverage in the village where I'm moving", "check_signal_coverage"),
 ("I want to cancel my subscription", "cancel_plan"),
 ("how much will it cost me to leave before the end of the contract", "check_cancellation_fee"),
 ("I want a bigger plan with more gigabytes", "change_plan"),
 ("I'm thinking of moving to another operator", "change_provider"),
 ("I'd like to sign up for the unlimited offer", "sign_up_for_plan"),
 ("stop the voicemail service please", "deactivate_call_management_services"),
 ("I need to see what I paid with my mobile last week", "check_mobile_payments"),
]
hard_texts = [t for t, _ in HARD]; hard_gold = [l for _, l in HARD]
assert all(l in label2id for l in hard_gold), [l for l in hard_gold if l not in label2id]
hard_pred = [id2label[i] for i in predict(hard_texts).argmax(1)]
hard_acc = float(np.mean([p == g for p, g in zip(hard_pred, hard_gold)]))
print(f"HAND-WRITTEN accuracy {hard_acc:.3f} ({sum(p == g for p, g in zip(hard_pred, hard_gold))}/{len(HARD)})")
for t, g, p in zip(hard_texts, hard_gold, hard_pred):
    if p != g: print(f"  MISS  {t!r}: expected {g}, got {p}")

HAND-WRITTEN accuracy 0.667 (20/30)
  MISS  'I deserve a refund for the days without service': expected get_compensation, got pay
  MISS  'no network at all in my house since Monday': expected report_poor_signal_coverage, got check_signal_coverage
  MISS  "calls keep dropping when I'm at work": expected report_poor_signal_coverage, got activate_call_management_services
  MISS  'my internet stopped working this morning': expected report_problem, got cancel_plan
  MISS  'why did I pay extra for data this month': expected check_excess_data_charges, got check_usage
  MISS  'I want to pay my bill now': expected pay, got invoices
  MISS  'my phone was stolen, block my line': expected deactivate_phone, got cancel_plan
  MISS  'I want a bigger plan with more gigabytes': expected change_plan, got sign_up_for_plan
  MISS  "I'm thinking of moving to another operator": expected change_provider, got human_agent
  MISS  'I need to see what I paid with my mobile last week': expected check_mobile_paym

MD## Baseline for comparison — TF-IDF + logistic regression (what we could do without deep learning)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
base = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), LogisticRegression(max_iter=2000, C=5))
base.fit(train_df.instruction, train_df.label)
b_test = base.predict(test_df.instruction)
b_hard = [id2label[i] for i in base.predict(hard_texts)]
baseline = dict(test_macro_f1=float(f1_score(test_df.label, b_test, average="macro")),
                hand_written_accuracy=float(np.mean([p == g for p, g in zip(b_hard, hard_gold)])))
print("TF-IDF baseline:", baseline)

TF-IDF baseline: {'test_macro_f1': 0.9919063073192979, 'hand_written_accuracy': 0.5666666666666667}


MD## Export

In [9]:
EXPORT = OUT / "advisor_intent_model"
shutil.rmtree(EXPORT, ignore_errors=True)
model.save_pretrained(EXPORT); tok.save_pretrained(EXPORT)
cat = df.drop_duplicates("intent").set_index("intent").category.to_dict()
metrics = dict(model=BASE, task="telecom intent classification", labels=labels, intent_category=cat,
               train_rows=len(train_df), val_rows=len(val_df), test_rows=len(test_df),
               test_accuracy=float(acc), test_macro_f1=float(f1), hand_written_accuracy=hard_acc,
               hand_written_examples=len(HARD), baseline_tfidf=baseline,
               data="bitext/Bitext-telco-llm-chatbot-training-dataset (CDLA-Sharing-1.0)", max_len=MAX_LEN, epochs=EPOCHS, seed=SEED)
(EXPORT / "metrics.json").write_text(json.dumps(metrics, indent=2))
(EXPORT / "MODEL_CARD.md").write_text(f'''# Advisor intent model
Fine-tuned {BASE} on the Bitext telco customer-support dataset (26 intents, CDLA-Sharing-1.0, attribution: Bitext).
Test macro-F1 {f1:.3f} (same distribution, optimistic). Hand-written messages: {hard_acc:.0%} of {len(HARD)}.
TF-IDF baseline: test macro-F1 {baseline["test_macro_f1"]:.3f}, hand-written {baseline["hand_written_accuracy"]:.0%}.
Limits: synthetic template-based English data; no real CallVerse calls; English only.
''')
shutil.make_archive(str(OUT / "advisor_intent_model"), "zip", EXPORT)
print("Download:", OUT / "advisor_intent_model.zip", f"({(OUT / 'advisor_intent_model.zip').stat().st_size/1e6:.0f} MB)")
print(json.dumps({k: v for k, v in metrics.items() if k not in ("labels", "intent_category")}, indent=2))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Download: /content/advisor_intent_model.zip (84 MB)
{
  "model": "sentence-transformers/all-MiniLM-L6-v2",
  "task": "telecom intent classification",
  "train_rows": 20788,
  "val_rows": 2599,
  "test_rows": 2599,
  "test_accuracy": 0.9996152366294728,
  "test_macro_f1": 0.9996153749997596,
  "hand_written_accuracy": 0.6666666666666666,
  "hand_written_examples": 30,
  "baseline_tfidf": {
    "test_macro_f1": 0.9919063073192979,
    "hand_written_accuracy": 0.5666666666666667
  },
  "data": "bitext/Bitext-telco-llm-chatbot-training-dataset (CDLA-Sharing-1.0)",
  "max_len": 64,
  "epochs": 4,
  "seed": 42
}


---
# PART 2 — Complaints models (run on the same Colab server)

# Complaints agent — escalation and severity models (real complaints)

**What it learns, from 100 000+ real consumer complaints (US CFPB database):**
1. **Escalation** — will the customer *dispute* the company's answer? (`Consumer disputed?` = Yes) → the agent's *escalation risk*.
2. **Severity** — will the case end with *money paid back* to the customer? (`Company Response = Closed with monetary relief`) → the agent's *severity*.

**Model:** TF-IDF + logistic regression. Chosen on purpose: every prediction can be explained word by word (our explainable-AI promise), it runs anywhere on CPU, and it is exported as plain JSON (no library-version problems).
**Data:** [milesbutler/consumer_complaints](https://huggingface.co/datasets/milesbutler/consumer_complaints) — the public CFPB Consumer Complaint Database (US public data). Domain = financial services, not telecom: we use it for *how* customers write when a complaint will escalate, not for telecom facts.

**Run:** CPU is enough (GPU useless here). ~10–20 min. Download **`complaints_models.zip`** at the end.

In [10]:
import os, json, re, random, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
# Kaggle writes to /kaggle/working, Lightning AI to the current folder
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
print("Outputs go to", OUT)

Outputs go to /content


In [11]:
from datasets import load_dataset
ds = load_dataset("milesbutler/consumer_complaints")
df = pd.concat([ds[s].to_pandas() for s in ds.keys()], ignore_index=True)
print(df.shape); print(list(df.columns))

def col(*words):
    """Find a column whatever its exact spelling ('Consumer disputed?' / 'consumer_disputed')."""
    norm = {c: re.sub(r"[^a-z]", "", c.lower()) for c in df.columns}
    for c, n in norm.items():
        if all(w in n for w in words): return c
    raise KeyError(words)

TEXT, DISPUTED, RESPONSE = col("complaint", "consumer") if "Consumer Complaint" not in df.columns else "Consumer Complaint", col("disputed"), col("response", "consumer")
print("text:", TEXT, "| disputed:", DISPUTED, "| response:", RESPONSE)
df = df[df[TEXT].notna()].copy()
df["text"] = (df[TEXT].astype(str)
              .str.replace(r"X{2,}", " ", regex=True)            # CFPB redactions: XXXX
              .str.replace(r"\{\$[\d.,]+\}", " AMOUNT ", regex=True)
              .str.replace(r"\s+", " ", regex=True).str.strip())
df = df[df.text.str.len() >= 30]
print(len(df), "complaints with a narrative")
print(df[DISPUTED].value_counts(dropna=False).head()); print(df[RESPONSE].value_counts(dropna=False).head(10))

README.md:   0%|          | 0.00/171 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/2.05k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 42.4MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/208398 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/69416 [00:00<?, ? examples/s]

(277814, 19)
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer Complaint', 'Company Public Response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date Sent to Company', 'Company Response to Consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'Unnamed: 18']
text: Consumer Complaint | disputed: Consumer disputed? | response: Company Response to Consumer
277259 complaints with a narrative
Consumer disputed?
No      128036
None    113438
Yes      35785
Name: count, dtype: int64
Company Response to Consumer
Closed with explanation            223343
Closed with non-monetary relief     31567
Closed with monetary relief         16651
Closed                               3735
Untimely response                    1961
None                                    1
In progress                             1
Name: count, dtype: int64


## The two targets

In [12]:
esc = df[df[DISPUTED].isin(["Yes", "No"])].copy()
esc["y"] = (esc[DISPUTED] == "Yes").astype(int)

resp = df[RESPONSE].fillna("")
sev = df[resp.str.startswith("Closed")].copy()
r = sev[RESPONSE].str.lower()
sev["y"] = (r.str.contains("monetary relief") & ~r.str.contains("non-monetary")).astype(int)

for name, d in [("escalation (disputed)", esc), ("severity (monetary relief)", sev)]:
    print(f"{name}: {len(d)} rows, positives {d.y.mean():.1%}")

escalation (disputed): 163821 rows, positives 21.8%
severity (monetary relief): 275296 rows, positives 6.0%


## Training: TF-IDF + logistic regression, class-balanced, 80/20 stratified split

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

MAX_ROWS = 200_000
VEC = dict(ngram_range=(1, 2), min_df=5, max_df=0.5, max_features=20_000, sublinear_tf=True, stop_words="english", lowercase=True)

def train(d, name):
    d = d.sample(min(len(d), MAX_ROWS), random_state=SEED)
    tr, te = train_test_split(d, test_size=0.2, stratify=d.y, random_state=SEED)
    vec = TfidfVectorizer(**VEC)
    Xtr, Xte = vec.fit_transform(tr.text), vec.transform(te.text)
    clf = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0, solver="liblinear").fit(Xtr, tr.y)
    p = clf.predict_proba(Xte)[:, 1]
    prec, rec, thr = precision_recall_curve(te.y, p)
    f1s = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
    best = int(np.nanargmax(f1s[:-1]))
    m = dict(rows_train=len(tr), rows_test=len(te), positive_rate=float(te.y.mean()),
             roc_auc=float(roc_auc_score(te.y, p)), pr_auc=float(average_precision_score(te.y, p)),
             best_threshold=float(thr[best]), f1_at_best=float(f1s[best]), f1_at_0_5=float(f1_score(te.y, p >= 0.5)))
    print(f"\n== {name}"); print(json.dumps(m, indent=2))
    terms = np.array(vec.get_feature_names_out()); w = clf.coef_[0]
    print("words that RAISE it :", ", ".join(terms[np.argsort(w)[-25:][::-1]]))
    print("words that LOWER it :", ", ".join(terms[np.argsort(w)[:25]]))
    return vec, clf, m

esc_vec, esc_clf, esc_m = train(esc, "ESCALATION (will the customer dispute?)")
sev_vec, sev_clf, sev_m = train(sev, "SEVERITY (will money be paid back?)")


== ESCALATION (will the customer dispute?)
{
  "rows_train": 131056,
  "rows_test": 32765,
  "positive_rate": 0.21843430489851975,
  "roc_auc": 0.6351537428244466,
  "pr_auc": 0.33606576609491756,
  "best_threshold": 0.4330808997775861,
  "f1_at_best": 0.395900799344128,
  "f1_at_0_5": 0.3854723027620937
}
words that RAISE it : scottrade, tcpa, equifax, key, debt listed, rent recovery, executive office, payments called, cfpb, target, usury, refuses, credit negative, failed, years trying, sending money, totalling, filed county, locks, violation, loan balance, investigation results, discover, deferral, end loan
words that LOWER it : dispute letters, requests investigate, nd, online banking, ic, letter disputing, did provide, unauthorized fraudulent, insurance time, hour, bankruptcy filing, balance closed, transworld, access funds, netspend, th, payment online, mothers, costco, account authorized, needs stop, called requested, 2016 paid, 3rd, unfair practices

== SEVERITY (will money be 

## Sanity check on telecom-style complaints (like the ones our agent receives)

In [14]:
TELECOM = [
 "I was charged twice for my mobile plan and nobody answers, I will go to the regulator and post this everywhere",
 "My internet has been down for a week, I called five times, I demand a refund for every day without service",
 "Small question about my invoice, the amount seems a bit higher than usual",
 "You promised a credit last month and it never came, this is fraud, my lawyer will contact you",
 "Thank you for fixing the roaming issue, I just want a copy of the invoice",
]
for t in TELECOM:
    print(f"escalation {esc_clf.predict_proba(esc_vec.transform([t]))[0,1]:.2f}  severity {sev_clf.predict_proba(sev_vec.transform([t]))[0,1]:.2f}  | {t[:80]}")

escalation 0.33  severity 0.28  | I was charged twice for my mobile plan and nobody answers, I will go to the regu
escalation 0.22  severity 0.31  | My internet has been down for a week, I called five times, I demand a refund for
escalation 0.61  severity 0.19  | Small question about my invoice, the amount seems a bit higher than usual
escalation 0.34  severity 0.12  | You promised a credit last month and it never came, this is fraud, my lawyer wil
escalation 0.25  severity 0.23  | Thank you for fixing the roaming issue, I just want a copy of the invoice


## Export as plain JSON (vocabulary, idf, weights) — rebuilt in the project with any scikit-learn version

In [15]:
EXPORT = OUT / "complaints_models"; shutil.rmtree(EXPORT, ignore_errors=True); EXPORT.mkdir()
def dump(vec, clf, m, name, target):
    json.dump(dict(target=target, vectorizer={k: (list(v) if isinstance(v, tuple) else v) for k, v in VEC.items()},
                   vocabulary={t: int(i) for t, i in vec.vocabulary_.items()}, idf=[float(x) for x in vec.idf_],
                   coef=[float(x) for x in clf.coef_[0]], intercept=float(clf.intercept_[0]), metrics=m,
                   data="milesbutler/consumer_complaints (US CFPB Consumer Complaint Database, public)"),
              open(EXPORT / f"{name}.json", "w"))
dump(esc_vec, esc_clf, esc_m, "escalation", "Consumer disputed? = Yes")
dump(sev_vec, sev_clf, sev_m, "severity", "Company response = Closed with monetary relief")
(EXPORT / "MODEL_CARD.md").write_text(f'''# Complaints models (CFPB)
Escalation: ROC-AUC {esc_m["roc_auc"]:.3f}, PR-AUC {esc_m["pr_auc"]:.3f} (positives {esc_m["positive_rate"]:.1%}).
Severity: ROC-AUC {sev_m["roc_auc"]:.3f}, PR-AUC {sev_m["pr_auc"]:.3f} (positives {sev_m["positive_rate"]:.1%}).
TF-IDF (1-2 grams, 20k terms) + class-balanced logistic regression, 80/20 stratified split.
Limits: US financial complaints, English; proxies (dispute, monetary relief) for escalation and severity; not telecom-specific.
''')
shutil.make_archive(str(OUT / "complaints_models"), "zip", EXPORT)
print("Download:", OUT / "complaints_models.zip", f"({(OUT / 'complaints_models.zip').stat().st_size/1e6:.1f} MB)")

Download: /content/complaints_models.zip (0.8 MB)


In [16]:
# PART 3 — export the advisor model to ONNX (int8) so the project runs it with onnxruntime, without torch
!pip install -q onnx onnxruntime
import torch, onnxruntime as ort, zipfile
from onnxruntime.quantization import quantize_dynamic, QuantType
P = OUT / "advisor_intent_model"
m = AutoModelForSequenceClassification.from_pretrained(P).eval(); t = AutoTokenizer.from_pretrained(P)
names = ["input_ids", "attention_mask", "token_type_ids"]
enc = t(["my bill is wrong"], return_tensors="pt")
kw = dict(input_names=names, output_names=["logits"], opset_version=17,
          dynamic_axes={**{k: {0: "batch", 1: "seq"} for k in names}, "logits": {0: "batch"}})
try:
    torch.onnx.export(m, tuple(enc[k] for k in names), str(P / "model_fp32.onnx"), dynamo=False, **kw)
except TypeError:
    torch.onnx.export(m, tuple(enc[k] for k in names), str(P / "model_fp32.onnx"), **kw)
quantize_dynamic(str(P / "model_fp32.onnx"), str(P / "model.onnx"), weight_type=QuantType.QInt8)

def onnx_pred(path, texts):
    s = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    out = []
    for i in range(0, len(texts), 256):
        e = t(list(texts[i:i + 256]), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="np")
        out.append(s.run(None, {k: e[k].astype("int64") for k in names})[0])
    return np.vstack(out).argmax(1)

q_test = onnx_pred(P / "model.onnx", test_df.instruction.values)
q_hard = [id2label[i] for i in onnx_pred(P / "model.onnx", hard_texts)]
onnx_metrics = dict(test_macro_f1_int8=float(f1_score(test_df.label, q_test, average="macro")),
                    hand_written_accuracy_int8=float(np.mean([p == g for p, g in zip(q_hard, hard_gold)])),
                    agreement_with_pytorch_test=float(np.mean(q_test == test_pred)),
                    size_mb=round((P / "model.onnx").stat().st_size / 1e6, 1))
print(json.dumps(onnx_metrics, indent=2))
meta = json.loads((P / "metrics.json").read_text()); meta["onnx_int8"] = onnx_metrics; (P / "metrics.json").write_text(json.dumps(meta, indent=2))
with zipfile.ZipFile(OUT / "advisor_intent_onnx.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["model.onnx", "tokenizer.json", "tokenizer_config.json", "config.json", "metrics.json", "MODEL_CARD.md"]:
        z.write(P / f, f)
print("Download:", OUT / "advisor_intent_onnx.zip", round((OUT / "advisor_intent_onnx.zip").stat().st_size / 1e6, 1), "MB")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

/tmp/ipykernel_1907/980573676.py:12: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(m, tuple(enc[k] for k in names), str(P / "model_fp32.onnx"), dynamo=False, **kw)
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:208: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.13/dist-packages/transformers/integratio

{
  "test_macro_f1_int8": 0.9996153749997596,
  "hand_written_accuracy_int8": 0.7333333333333333,
  "agreement_with_pytorch_test": 1.0,
  "size_mb": 23.0
}
Download: /content/advisor_intent_onnx.zip 15.6 MB
